# Анализ блоков и ошибок Orthrus
Загружается финальный экспорт выбранного запуска. Для промежуточного чекпоинта задай `cfg.eval.checkpoint = "latest"`.
Запись через hooks замедляет генерацию: этот ноутбук предназначен для анализа, а скорость измеряется отдельным benchmark.

In [ ]:
from pathlib import Path
import os
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/config.py").exists())
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import load_config
CONFIG = os.environ.get("ORTHRUS_CONFIG", "configs/qwen3-0.6b.yaml")
cfg = load_config(CONFIG)
print("Experiment:", cfg.run_name)


In [ ]:
import torch
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
from src.test.benchmark import load_for_eval, chat_prompt, eos_token_ids
from src.test.trace import trace_orthrus, rejection_rows

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = getattr(torch, cfg.eval.dtype) if device.type == "cuda" else torch.float32
model, tokenizer, label = load_for_eval(cfg, device, dtype)
eos = eos_token_ids(model, tokenizer)
print("Loaded:", label)


In [ ]:
PROMPT = "Explain to a five-year-old why the sky is blue."
MAX_NEW_TOKENS = 256
input_ids = torch.tensor([chat_prompt(tokenizer, PROMPT)], device=device)
full_ids, blocks, stats = trace_orthrus(model, input_ids, MAX_NEW_TOKENS, sorted(eos))
print(tokenizer.decode(full_ids[input_ids.shape[1]:], skip_special_tokens=True))
print("Blocks:", len(blocks), "TPF:", stats.new_tokens / stats.forward_passes)


## Просмотр блока
Контекст начинается с начала сгенерированного ответа. AR-предсказания после первого отказа условны на уже неверном черновике.

In [ ]:
def token_text(token):
    return repr(tokenizer.decode([token], skip_special_tokens=False))

def show_block(number):
    block = blocks[number - 1]
    start, accepted = block["start"], block["accepted"]
    context = full_ids[max(input_ids.shape[1], start - 24):start]
    print("Context:", tokenizer.decode(context, skip_special_tokens=True))
    print("Anchor:", token_text(block["anchor"]))
    print("Accepted:", " | ".join(token_text(t) for t in block["draft"][:accepted]))
    rows = []
    for j, token in enumerate(block["draft"]):
        state = "accepted" if j < accepted else "rejected" if j == accepted and block["rejected"] else "discarded"
        rows.append({"position": j + 1, "draft": token_text(token), "AR": token_text(block["ar"][j]),
                     "status": state,
                     "diffusion top-4": " | ".join(f"{token_text(t)}: {p:.6f}" for t, p in zip(block["top_ids"][j], block["top_probs"][j]))})
    display(pd.DataFrame(rows))

if blocks:
    widgets.interact(show_block, number=widgets.IntSlider(min=1, max=len(blocks), value=1))
else:
    print("Generation ended before the first diffusion block.")


## Первые отказы блоков
Флаги repetition/skip/first_rejected/part_of_token могут пересекаться. `skip` проверяется по следующему токену фактического ответа; если генерация обрезана ровно после отказа, следующего токена нет и этот признак не определяется.

In [ ]:
rows = rejection_rows(blocks, full_ids, tokenizer, eos)
errors = pd.DataFrame(rows)
if not errors.empty:
    errors = errors.sort_values("ratio", ascending=False, ignore_index=True)
    with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
        display(errors)
    flags = ["repetition", "skip", "first_rejected", "part_of_token", "second_is_correct", "other"]
    summary = pd.DataFrame({"count": errors[flags].sum(), "percent": errors[flags].mean() * 100})
    display(summary)
    destination = cfg.run_path / "analysis"
    destination.mkdir(parents=True, exist_ok=True)
    errors.to_csv(destination / "errors.csv", index=False, encoding="utf-8-sig")
    print("Saved:", destination / "errors.csv")
else:
    print("No rejections before the end of this answer.")


Чтобы сравнить несколько промптов, повтори вызовы `trace_orthrus` и `rejection_rows`, добавляя к строкам имя примера или домена. Не используй trace для замера скорости.